In [1]:
import torch
from peft import PeftModel
from transformers import AutoTokenizer, EsmModel

# Use your specific model names
base_model_path = "facebook/esm2_t33_650M_UR50D"
adapter_path = "AmelieSchreiber/esm2_t33_650M_qlora_binding_16M"

try:
    import bitsandbytes as bnb
    print(f"✅ bitsandbytes found! Version: {bnb.__version__}")
    
    base_model = EsmModel.from_pretrained(base_model_path)
    model = PeftModel.from_pretrained(base_model, adapter_path)
    
    # CHECK: Is it actually a PeftModel?
    if isinstance(model, PeftModel):
        print("🚀 SUCCESS: QBind adapter integrated. You are using the 1280-dim QBind version.")
    else:
        print("⚠️ Still using Vanilla ESM-2.")
        
except ModuleNotFoundError:
    print("❌ bitsandbytes still missing. Install it to use the QBind adapter.")
except Exception as e:
    print(f"❌ Error: {e}")

✅ bitsandbytes found! Version: 0.49.2


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 SUCCESS: QBind adapter integrated. You are using the 1280-dim QBind version.


/home/va.melnyk@ad.loc/anaconda3/envs/protacs/lib/python3.13/site-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.encoder.layer.0.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.0.attention.self.key.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.key.lora_B.default.weight', 'base_model.model.encoder.layer.0.attention.self.value.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.value.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.1.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self.key.lora_A.default.weight', 'base_model.model.encoder.layer.1.attention.self.key.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self

In [2]:
import torch
import transformers
import peft
import bitsandbytes as bnb # Required for QLoRA

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PEFT version: {peft.__version__}")
print(f"BitsAndBytes version: {bnb.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")

PyTorch version: 2.8.0
Transformers version: 5.2.0
PEFT version: 0.18.1
BitsAndBytes version: 0.49.2
CUDA Available: False


In [3]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Load the tokenizer and model
# Example: ChemBERTa-2 trained on 77M SMILES via MLM
model_name = "deepchem/ChemBERTa-77M-MLM" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Prepare your SMILES
smiles = ["CC(=O)Oc1ccccc1C(=O)O", "CCO"] # Aspirin and Ethanol

# 3. Tokenize and generate hidden states
inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

# 4. Extract Embeddings
# Option A: [CLS] token (standard for ChemBERTa)
# The [CLS] token is at index 0 for each sequence
cls_embeddings = outputs.last_hidden_state[:, 0, :]

# Option B: Mean Pooling (averaging all token embeddings)
attention_mask = inputs['attention_mask']
token_embeddings = outputs.last_hidden_state
input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
mean_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

print(f"Embedding shape: {cls_embeddings.shape}") # [batch_size, 768]
print(f"Mean Pooling Embedding shape: {mean_embeddings.shape}")

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: deepchem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Embedding shape: torch.Size([2, 384])
Mean Pooling Embedding shape: torch.Size([2, 384])
